# 03 — Quantization v2: calibrated INT8

Produces `models512_int8/` — a **mixed** engine set, INT8 where it is safe and
fp16 where it is not — plus the numbers that justify each choice.

> Version 1 of this question is already answered in `docs/tensorrt_fp16.md`, as
> a PyTorch *simulation*. It concluded INT8 was calibration-sensitive rather
> than impossible. This notebook builds the real thing.

## The finding this is built on

INT8 damage is **not uniform across EdgeTAM**, and neither is the right
calibrator. A single global setting is why a first attempt looks impossible.

| module | share of FLOPs | absmax (TensorRT's default) | percentile-clipped |
|---|---:|---:|---:|
| memory attention | **61.9 %** | 0.9998 | 0.9998 |
| memory encoder | 8.8 % | **0.9999** | 0.9397 |
| SAM head | 2.5 % | 0.9993 | 0.9990 |
| image encoder | 26.8 % | **0.8170** | 0.9936 |

Two entries decide everything, and they point opposite ways:

- **The image encoder collapses under min/max.** RepViT is a depthwise-separable
  stack whose per-channel ranges differ by orders of magnitude, while TensorRT
  scales activations *per tensor*. A few outliers stretch the scale until the
  bulk of the distribution has no resolution left. → **entropy**.
- **The memory encoder is the reverse — clipping hurts it.** And the shape of
  the failure names the mechanism: rounding scatters, clipping is *systematic*.
  The memory encoder's output **is** the stored memory, so a systematic bias
  goes into the bank and is read back for seven frames. → **max, never clipped**.
- **The memory attention is 62 % of the arithmetic and barely notices.** It only
  *reads* the bank, so it inherits none of the accumulation problem. This is
  the best ratio in the model and the reason the exercise is worth doing.

## PTQ, QAT, distillation — the actual division of labour

| | used for |
|---|---|
| **PTQ** | the default, and usually the end of it |
| **QAT** | only a module PTQ leaves short — in practice the image encoder |
| **distillation** | twice, for unrelated reasons: SAM 2.1-L → EdgeTAM for *labels* (notebook 01), and fp16 → INT8 during QAT |

In [ ]:
import os, sys
from pathlib import Path

REPO = Path("/content/sam-dedection")
if not REPO.exists():
    !git clone -q https://github.com/yigitkayabagci/sam-dedection.git {REPO}
os.chdir(REPO); sys.path.insert(0, str(REPO))

!bash scripts/setup_edgetam.sh 2>&1 | tail -3
!pip install -q -r requirements.txt nvidia-modelopt[onnx] onnx onnxruntime
!python -m unittest tests.test_quantization 2>&1 | tail -3

In [ ]:
# --- Data on local disk, keepsakes on Drive -----------------------------
# The dataset is a few hundred thousand small JPEGs; the Drive FUSE mount
# serves those an order of magnitude slower than the GPU reads them. Drive
# holds only what is worth surviving the runtime: the checkpoint and the RLE
# label store, both megabytes.
DATA_DIR = Path("/content/data")
!python tools/fetch_antiuav410.py --dest {DATA_DIR} --splits train val

from tools.fetch_antiuav410 import dataset_root, describe, find_splits

splits = find_splits(DATA_DIR)
DATA = dataset_root(splits)
print(describe(splits))

WORK = Path("/content/work")
try:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/edgetam-thermal")
except Exception as exc:
    print(f"no Drive ({type(exc).__name__}) -- WORK stays at {WORK}")
WORK.mkdir(parents=True, exist_ok=True)

CKPT = REPO / "checkpoints"; CKPT.mkdir(exist_ok=True)
SIZE = 512

# Calibrate the checkpoint you will deploy. Scales taken from the stock model
# would be calibrated for activation distributions the fine-tune moved.
import shutil
tuned = WORK / "edgetam_thermal_512.pt"
assert tuned.is_file(), (f"{tuned} is not there -- run notebook 02 first, or copy "
                         f"its checkpoint into {WORK}")
shutil.copy(tuned, CKPT / "edgetam_thermal_512.pt")

In [ ]:
# --- The plan, before anything is built --------------------------------
!python tools/quantize_edgetam.py plan

## Step 1 — export the fp32 graphs

Unchanged from the existing pipeline. The fine-tune moved weights, never
architecture, so `export_edgetam_onnx.py` needs no argument it did not already
have.

In [ ]:
!python tools/export_edgetam_onnx.py --outdir models512/ --image-size {SIZE} \
    --checkpoint checkpoints/edgetam_thermal_512.pt --verify 2>&1 | tail -20

## Step 2 — capture real calibration data

The image encoder is the easy one: feed it thermal frames. The other three take
`pix_feat`, a padded `memory` bank and a sigmoid-scaled `mask_for_mem` — nobody
can write those distributions down, and calibrating on noise sets scales for a
network that does not exist.

So this tracks real sequences through **PyTorch stand-ins for the engines**
(`tests/reference_engines.py` — the same wrapper graphs the ONNX came from,
behind the same runtime API) and records what each one is fed, in exactly the
layout the deployed engine receives. No TensorRT required, which matters:
calibration has to happen *before* an INT8 engine exists.

Samples are **strided across the clip, not taken from the opening frames**. The
memory bank fills over the first ~16 frames; calibrating on that transient
tunes for a state the engine spends almost none of its time in.

In [ ]:
!python tools/quantize_edgetam.py capture --data {DATA} --split train \
    --calib outputs/calib/ --checkpoint checkpoints/edgetam_thermal_512.pt \
    --image-size {SIZE} --device cuda --sequences 8 --frames 64 \
    --limit 512 --stride 2 2>&1 | tail -12

In [ ]:
# --- Look at what was captured -----------------------------------------
# The distribution shape IS the calibration decision. A long right tail with a
# far heavier absmax than p99.9 is exactly the situation min/max calibration
# handles badly -- and it should be visible here, for the image encoder.
import numpy as np
import matplotlib.pyplot as plt

MODULES = ["image_encoder", "memory_attention", "memory_encoder", "sam_head"]
fig, axes = plt.subplots(1, len(MODULES), figsize=(4 * len(MODULES), 3.2))
print(f"{'module':<18}{'tensor':<14}{'p99.9':>10}{'absmax':>10}{'ratio':>9}")
for ax, name in zip(axes, MODULES):
    with np.load(f"outputs/calib/{name}.npz") as data:
        key = max(data.files, key=lambda k: data[k].size)
        values = np.abs(data[key].ravel())
    values = values[values > 0]
    p999, top = np.percentile(values, 99.9), values.max()
    print(f"{name:<18}{key:<14}{p999:>10.3f}{top:>10.3f}{top / max(p999, 1e-9):>9.1f}x")
    ax.hist(np.log10(values + 1e-9), bins=80, color="#2a78d6")
    ax.axvline(np.log10(p999), color="#eb6834", lw=1.2, label="p99.9")
    ax.axvline(np.log10(top), color="#4a3aa7", lw=1.2, label="absmax")
    ax.set_title(f"{name}\n{key}", fontsize=9); ax.set_xlabel("log10 |x|")
    ax.legend(fontsize=7)
plt.tight_layout(); plt.show()
print("\nA large absmax/p99.9 ratio is the signature min/max calibration cannot "
      "handle: the scale is set by a handful of values and the rest of the "
      "distribution loses its resolution.")

## Step 3 — PTQ

Model Optimizer, per module, with the calibrator each module's own measurement
asked for. Output is a self-contained directory: modules held at fp16 are
copied in unchanged, and each spec records the precision to build it at — so
one build command produces the whole mixed set.

In [ ]:
!python tools/quantize_edgetam.py quantize --outdir models512/ \
    --qdq-outdir models512_int8/ --calib outputs/calib/ --image-size {SIZE} 2>&1 | tail -25

In [ ]:
# --- Did the graphs actually get Q/DQ nodes? ---------------------------
# The cheapest possible check that PTQ did something, and it catches the
# failure where quantize() silently no-ops on an unsupported graph.
import json, onnx

for name in MODULES:
    path = Path("models512_int8") / f"edgetam_{name}.onnx"
    graph = onnx.load(str(path)).graph
    qdq = sum(n.op_type in ("QuantizeLinear", "DequantizeLinear") for n in graph.node)
    spec = json.loads(path.with_suffix(".spec.json").read_text())
    print(f"{name:<18} {spec['precision']:<5} {len(graph.node):>5} nodes, "
          f"{qdq:>4} Q/DQ")
    if spec["precision"] == "int8":
        assert qdq > 0, f"{name}: marked int8 but has no Q/DQ nodes"

## Step 4 — does it still track?

`check_trt_parity.py` compares each engine against its module in isolation,
with a fresh input every call. That **cannot see accumulation**, and EdgeTAM
feeds its own masks back through the memory bank — which is exactly where the
memory encoder's clipping bias showed up in the fp16 study.

So the number that decides anything is state accuracy over whole sequences.
Both are run below; if they disagree, believe the second one.

In [ ]:
# Engines are GPU- and TensorRT-version-specific: these must be built on the
# Orin, not here. Left as the exact commands to run there.
print(f'''On the Orin:

  python tools/build_trt_engines.py --outdir models512_int8/ \\
      --precision auto --max-batch 4
  python tools/check_trt_parity.py --outdir models512_int8/ --image-size {SIZE}

  # fp16 reference
  python tools/eval_antiuav.py --data <anti-uav410> --split val --limit 20 \\
      --tracker edgetam_trt --config configs/edgetam_trt_512.yaml \\
      --json results/fp16.json
  # INT8 candidate
  python tools/eval_antiuav.py --data <anti-uav410> --split val --limit 20 \\
      --tracker edgetam_trt --config configs/edgetam_trt_int8_512.yaml \\
      --json results/int8.json

Gate, per module: accept INT8 if state accuracy drops by at most 1.0 point and
mask IoU against the fp16 run stays above 0.99. A module that fails goes back
to fp16 on its own -- every engine is independent:

  python tools/quantize_edgetam.py quantize --outdir models512/ \\
      --qdq-outdir models512_int8/ --calib outputs/calib/ --fp16 image_encoder
''')

In [ ]:
# --- Simulate the gate here, before spending Orin time -----------------
# analyze_precision.py's INT8 model is pessimistic (activations only, weights
# untouched, TensorRT quantises those per channel and is more forgiving). A
# module that clears the bar here will clear it there.
!python tools/analyze_precision.py --frames 30 --skip-precision --verify-graphs \
    --checkpoint checkpoints/edgetam_thermal_512.pt 2>&1 | tail -12

## Step 5 — QAT, only if PTQ left something short

Skip this entirely if every module passed. QAT exists here for one likely
case — the image encoder — and is deliberately unavailable for the memory path:
`insert_quantizers` refuses `memory_attention` and `memory_encoder` unless
asked twice, for the same reason `finetune.py` freezes them. If a memory-path
module fails its gate, the honest answer is to leave that engine at fp16.

The teacher is **the same network before quantisation**, not a bigger model:
the target is to reproduce fp16 exactly. That is why a short schedule works —
the weights only have to move far enough to absorb rounding — and why the loss
is a KL, so that **zero means identical**.

In [ ]:
QAT_MODULES = []          # e.g. ["image_encoder"] if it failed the gate
QAT_EPOCHS = 1            # ~10% of the fine-tune schedule, per NVIDIA's guidance

if not QAT_MODULES:
    print("PTQ was enough -- nothing to do. Set QAT_MODULES to run this.")
else:
    import copy, json, torch
    from sam2.build_sam import build_sam2_video_predictor
    from src.trackers._hydra_overrides import image_size_overrides
    from src.training import list_sequences, open_masks, sample_clips
    from src.training.clip_loop import collate, propagate
    from src.training.finetune import Rates, apply_freeze, param_groups, save_checkpoint
    from src.training.qat import calibration_loop, distillation_loss, insert_quantizers
    from tqdm.auto import tqdm

    manifest = json.loads((WORK / "manifest.json").read_text())
    sequences = [s for s in list_sequences(DATA, "train")
                 if s.name in set(manifest["sequences"]["train"])]
    stores = {s.name: open_masks(WORK / "labels" / "train" / s.name /
                                 "pseudo_masks.npz") for s in sequences}
    clips = sample_clips(sequences, length=manifest["clip"]["length"],
                         stride=manifest["clip"]["stride"], size=SIZE,
                         frame_size=tuple(manifest["frame_size"]), seed=0)
    batches = [collate(clips[i:i + 2], [stores[c.sequence.name] for c in clips[i:i + 2]],
                       "cuda") for i in range(0, min(len(clips), 64), 2)]

    build = lambda: build_sam2_video_predictor(
        "configs/edgetam.yaml", "checkpoints/edgetam_thermal_512.pt", device="cuda",
        hydra_overrides_extra=image_size_overrides(SIZE)).eval()
    teacher = build()
    for p in teacher.parameters():
        p.requires_grad_(False)

    student = build()
    insert_quantizers(student, QAT_MODULES, calibration_loop(student, batches[:8]))
    apply_freeze(student, "encoder")
    opt = torch.optim.AdamW(param_groups(student, Rates(head=1e-5, neck=1e-5, trunk=1e-6)))

    for epoch in range(QAT_EPOCHS):
        for batch in tqdm(batches, desc=f"qat e{epoch}"):
            with torch.no_grad():
                reference = propagate(teacher, batch.images, batch.boxes[:, 0])
            with torch.autocast("cuda", dtype=torch.bfloat16):
                outputs = propagate(student, batch.images, batch.boxes[:, 0])
                loss = sum(distillation_loss(s, t) for s, t in
                           zip(outputs[1:], reference[1:])) / max(len(outputs) - 1, 1)
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        print(f"epoch {epoch}: KL from the fp16 network {float(loss):.5f} "
              "(0 = identical)")

    save_checkpoint(student, CKPT / "edgetam_thermal_512_qat.pt",
                    {"qat_modules": QAT_MODULES, "image_size": SIZE})

## What you have, and how to read it

- `models512_int8/` — the mixed engine set's graphs, plus
  `quantization_plan.json` recording what was done to each module and why.
- `outputs/calib/*.npz` — the recorded activations. Keep them: re-quantising
  after a threshold change costs seconds instead of another tracking run.

**One cost, stated rather than buried.** INT8 builds are *strongly typed*, so
TensorRT obeys the calibrated Q/DQ nodes instead of re-deciding precision layer
by layer. But a strongly-typed engine also takes its IO types from the graph,
which the exporter writes as fp32 — so the fp16 boundary that
`--inputIOFormats/--outputIOFormats` bought is given back, and reformat kernels
return. Whether the INT8 compute wins by more than the boundary loses is a
measurement, and it is per module. `check_trt_parity.py` reports it.

**Next:** `04_samurai_long_video.ipynb`. It is independent of everything here —
training-free, no engine changes — and it targets the failure mode this project
actually hit rather than the one it profiled.